# Initial ML Solution - Resumable Colab Experiments

Runs four extensions on the official AVeriTeC train/dev split: (1) pure dense retrieval over each claim's full candidate pool, (2) BM25+dense reciprocal-rank fusion, (3) an evidence-aware transformer with BM25 top-three evidence, and (4) the same transformer with hybrid top-three evidence.

Use a Colab GPU runtime and Google Drive for persistent outputs. After an interruption, reconnect, rerun setup, then rerun the interrupted cell with the same run name and `--resume`. Dense retrieval checkpoints each completed claim; transformer tuning checkpoints every 100 optimizer steps.

The dev corpus contains about 16 million passages, so dense retrieval may take many hours. If embedding runs out of GPU memory, lower `--batch-size` and rerun: completed claims stay checkpointed and only the interrupted claim is recomputed.

Push the project changes to GitHub before running the bootstrap cell, because it clones the repository.

In [ ]:
from pathlib import Path
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/muyishen2040/10718-ML-In-Practice.git'
REPO_DIR = Path('/content/10718-ML-In-Practice')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
import json
DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
PROCESSED = DATA_ROOT / 'processed' / 'averitec'
RAW_ARCHIVES = DATA_ROOT / 'raw' / 'averitec'
OUTPUT_ROOT = DATA_ROOT / 'outputs' / 'initial_ml_solution'
HF_CACHE = DATA_ROOT / 'cache' / 'huggingface'
HF_CACHE.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['SENTENCE_TRANSFORMERS_HOME'] = str(HF_CACHE / 'sentence-transformers')
DEV_CORPUS = PROCESSED / 'dev_evidence_corpus.jsonl'
DEV_QRELS = PROCESSED / 'dev_qrels.jsonl'
TRAIN_CLAIMS = PROCESSED / 'train.jsonl'
DEV_CLAIMS = PROCESSED / 'dev.jsonl'
TRAIN_BM25 = DATA_ROOT / 'outputs' / 'bm25' / 'train_bm25_top20_v2' / 'train_rankings.jsonl'
DEV_BM25 = DATA_ROOT / 'outputs' / 'bm25' / 'dev_bm25_top20_v2' / 'dev_rankings.jsonl'
required = [DEV_CORPUS, DEV_QRELS, TRAIN_CLAIMS, DEV_CLAIMS, TRAIN_BM25, DEV_BM25]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing expected v2 inputs on Drive:\n' + '\n'.join(missing))
print('Data root:', DATA_ROOT)
print('Output root:', OUTPUT_ROOT)

In [ ]:
import subprocess
DENSE_MODEL = 'BAAI/bge-small-en-v1.5'
DENSE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
DENSE_DEV_DIR = OUTPUT_ROOT / 'dense' / 'dev_bge_small_v15'
DENSE_TRAIN_DIR = OUTPUT_ROOT / 'dense' / 'train_bge_small_v15'
HYBRID_DEV_DIR = OUTPUT_ROOT / 'hybrid' / 'dev_rrf60'
HYBRID_TRAIN_DIR = OUTPUT_ROOT / 'hybrid' / 'train_rrf60'
TRANSFORMER_DIR = OUTPUT_ROOT / 'transformer'
def run_command(command):
    print('Running:', ' '.join(str(part) for part in command))
    subprocess.run([str(part) for part in command], cwd=REPO_DIR, check=True)
def run_if_missing(command, outputs):
    outputs = [Path(path) for path in outputs]
    if all(path.exists() for path in outputs):
        print('Already complete; reusing:', ', '.join(str(path) for path in outputs))
    else:
        run_command(command)

## Experiment 1 - pure dense retrieval

The runner ranks every passage in each claim's candidate pool, keeps the top 20 for downstream use, and reports the user-facing URL-proxy hit@3 and MRR. The evidence relevance signal is an annotated-source-URL proxy.

In [ ]:
dense_dev_rankings = DENSE_DEV_DIR / 'dev_rankings.jsonl'
dense_dev_metrics = DENSE_DEV_DIR / 'dev_metrics.json'
dense_dev_manifest = DENSE_DEV_DIR / 'dev_manifest.json'
dense_dev_command = [
    'python', '-u', 'scripts/run_dense_retrieval.py',
    '--split', 'dev', '--data-root', DATA_ROOT, '--corpus', DEV_CORPUS, '--qrels', DEV_QRELS,
    '--output-dir', OUTPUT_ROOT / 'dense', '--run-name', 'dev_bge_small_v15',
    '--model', DENSE_MODEL, '--query-prefix', DENSE_QUERY_PREFIX,
    '--ranking-k', '20', '--metric-k', '3', '--batch-size', '64', '--max-seq-length', '384',
    '--checkpoint-every', '1', '--resume',
]
run_if_missing(dense_dev_command, [dense_dev_rankings, dense_dev_metrics, dense_dev_manifest])

## Experiment 2 - hybrid BM25 + dense ranking

Reciprocal rank fusion combines the BM25 and dense top-20 rankings with a fixed constant of 60. This is preselected rather than tuned against final dev results.

In [ ]:
hybrid_dev_rankings = HYBRID_DEV_DIR / 'hybrid_rankings.jsonl'
hybrid_dev_metrics = HYBRID_DEV_DIR / 'hybrid_metrics.json'
hybrid_dev_manifest = HYBRID_DEV_DIR / 'hybrid_manifest.json'
hybrid_dev_command = [
    'python', '-u', 'scripts/run_hybrid_retrieval.py',
    '--bm25-rankings', DEV_BM25, '--dense-rankings', dense_dev_rankings, '--qrels', DEV_QRELS,
    '--output-dir', OUTPUT_ROOT / 'hybrid', '--run-name', 'dev_rrf60',
    '--rrf-constant', '60', '--ranking-k', '20', '--metric-k', '3',
]
run_if_missing(hybrid_dev_command, [hybrid_dev_rankings, hybrid_dev_metrics, hybrid_dev_manifest])

## Experiment 3 - transformer verifier with BM25 evidence

The model receives each claim and the same BM25 top-three evidence as the logistic-regression baseline. Hyperparameter selection uses a stratified 10% holdout from official train only: two learning rates times two epoch counts (four trials). Macro-F1 selects the configuration, with false-reassurance and false-alarm rates as tie-breakers. The selected settings are refit on all 3,067 usable train claims and evaluated once on the unchanged 500-claim dev split.

Each trial saves periodic checkpoints to Drive. Rerun this cell with the same run name and `--resume` after reconnecting.

In [ ]:
transformer_bm25_dir = TRANSFORMER_DIR / 'bm25_top3'
transformer_bm25_metrics = transformer_bm25_dir / 'final_dev_metrics.json'
transformer_bm25_predictions = transformer_bm25_dir / 'final_dev_predictions.jsonl'
transformer_bm25_manifest = transformer_bm25_dir / 'transformer_manifest.json'
transformer_bm25_command = [
    'python', '-u', 'scripts/train_transformer_verifier.py',
    '--data-root', DATA_ROOT, '--train-rankings', TRAIN_BM25, '--eval-rankings', DEV_BM25,
    '--output-dir', TRANSFORMER_DIR, '--run-name', 'bm25_top3',
    '--model', 'distilbert-base-uncased', '--max-length', '512',
    '--claim-token-budget', '96', '--evidence-token-budget', '110',
    '--validation-fraction', '0.10', '--learning-rates', '2e-5,5e-5', '--epoch-grid', '2,3',
    '--batch-size', '8', '--gradient-accumulation-steps', '2', '--checkpoint-steps', '100',
    '--seed', '718', '--resume',
]
run_if_missing(transformer_bm25_command, [transformer_bm25_metrics, transformer_bm25_predictions, transformer_bm25_manifest])

## Experiment 4 - hybrid retrieval + transformer

To keep training and prediction evidence consistent, this experiment also creates dense top-20 rankings for each train claim, fuses them with the saved train BM25 rankings, and trains the transformer using hybrid top-three evidence. The train shards are large; the already-downloaded archive files should be on Drive. Each rerun skips claims already in the dense checkpoint.

In [ ]:
TRAIN_ARCHIVES = [
    RAW_ARCHIVES / 'train_0_999.zip',
    RAW_ARCHIVES / 'train_1000_1999.zip',
    RAW_ARCHIVES / 'train_2000_3067.zip',
]
missing_archives = [str(path) for path in TRAIN_ARCHIVES if not path.is_file()]
if missing_archives:
    raise FileNotFoundError('Missing archives on Drive:\n' + '\n'.join(missing_archives))
dense_train_command = [
    'python', '-u', 'scripts/run_dense_retrieval.py',
    '--split', 'train', '--data-root', DATA_ROOT,
]
for archive_path in TRAIN_ARCHIVES:
    dense_train_command.extend(['--archive', archive_path])
dense_train_command.extend([
    '--output-dir', OUTPUT_ROOT / 'dense', '--run-name', 'train_bge_small_v15',
    '--model', DENSE_MODEL, '--query-prefix', DENSE_QUERY_PREFIX,
    '--ranking-k', '20', '--metric-k', '3', '--batch-size', '64', '--max-seq-length', '384',
    '--checkpoint-every', '1', '--max-words', '160', '--overlap-words', '40', '--resume',
])
run_command(dense_train_command)
print('All archive shards processed; finalize the train ranking in the next cell.')

In [ ]:
dense_train_rankings = DENSE_TRAIN_DIR / 'train_rankings.jsonl'
dense_train_metrics = DENSE_TRAIN_DIR / 'train_metrics.json'
dense_train_manifest = DENSE_TRAIN_DIR / 'train_manifest.json'
if all(path.exists() for path in [dense_train_rankings, dense_train_metrics, dense_train_manifest]):
    print('Already finalized:', dense_train_rankings)
else:
    run_command([
        'python', '-u', 'scripts/run_dense_retrieval.py',
        '--split', 'train', '--data-root', DATA_ROOT,
        '--output-dir', OUTPUT_ROOT / 'dense', '--run-name', 'train_bge_small_v15',
        '--model', DENSE_MODEL, '--query-prefix', DENSE_QUERY_PREFIX,
        '--ranking-k', '20', '--metric-k', '3', '--batch-size', '64', '--max-seq-length', '384',
        '--resume', '--finalize', '--max-words', '160', '--overlap-words', '40',
    ])

In [ ]:
hybrid_train_rankings = HYBRID_TRAIN_DIR / 'hybrid_rankings.jsonl'
hybrid_train_metrics = HYBRID_TRAIN_DIR / 'hybrid_metrics.json'
hybrid_train_manifest = HYBRID_TRAIN_DIR / 'hybrid_manifest.json'
hybrid_train_command = [
    'python', '-u', 'scripts/run_hybrid_retrieval.py',
    '--bm25-rankings', TRAIN_BM25, '--dense-rankings', dense_train_rankings,
    '--output-dir', OUTPUT_ROOT / 'hybrid', '--run-name', 'train_rrf60',
    '--rrf-constant', '60', '--ranking-k', '20', '--metric-k', '3',
]
run_if_missing(hybrid_train_command, [hybrid_train_rankings, hybrid_train_metrics, hybrid_train_manifest])

In [ ]:
transformer_hybrid_dir = TRANSFORMER_DIR / 'hybrid_top3'
transformer_hybrid_metrics = transformer_hybrid_dir / 'final_dev_metrics.json'
transformer_hybrid_predictions = transformer_hybrid_dir / 'final_dev_predictions.jsonl'
transformer_hybrid_manifest = transformer_hybrid_dir / 'transformer_manifest.json'
transformer_hybrid_command = [
    'python', '-u', 'scripts/train_transformer_verifier.py',
    '--data-root', DATA_ROOT, '--train-rankings', hybrid_train_rankings, '--eval-rankings', hybrid_dev_rankings,
    '--output-dir', TRANSFORMER_DIR, '--run-name', 'hybrid_top3',
    '--model', 'distilbert-base-uncased', '--max-length', '512',
    '--claim-token-budget', '96', '--evidence-token-budget', '110',
    '--validation-fraction', '0.10', '--learning-rates', '2e-5,5e-5', '--epoch-grid', '2,3',
    '--batch-size', '8', '--gradient-accumulation-steps', '2', '--checkpoint-steps', '100',
    '--seed', '718', '--resume',
]
run_if_missing(transformer_hybrid_command, [transformer_hybrid_metrics, transformer_hybrid_predictions, transformer_hybrid_manifest])

## Compare results

This prints the baseline and new dev metrics. Interpret Macro-F1 together with false reassurance and false alarm rates; accuracy alone can hide majority-label behavior.

In [ ]:
def read_metrics(path):
    path = Path(path)
    return json.loads(path.read_text(encoding='utf-8')) if path.is_file() else None
result_paths = {
    'BM25 retrieval': DATA_ROOT / 'outputs' / 'bm25' / 'dev_bm25_top20_v2' / 'dev_metrics.json',
    'Dense retrieval': dense_dev_metrics,
    'Hybrid retrieval': hybrid_dev_metrics,
    'Qwen + BM25': DATA_ROOT / 'outputs' / 'local_llm' / 'dev_qwen3_4b_bm25_top3_v2' / 'classify_top3_metrics.json',
    'TF-IDF LR + BM25': DATA_ROOT / 'outputs' / 'tfidf_logreg' / 'retrieved_train_to_dev_v2' / 'retrieved_train_to_dev_metrics.json',
    'Transformer + BM25': transformer_bm25_metrics,
    'Transformer + hybrid': transformer_hybrid_metrics,
}
for name, path in result_paths.items():
    result = read_metrics(path)
    if result is None:
        print(name, ': not run yet')
        continue
    errors = result.get('decision_error_rates', {})
    print(name, {
        'URL_proxy_hit@3': result.get('evidence_hit_at_3'),
        'MRR': result.get('mrr'),
        'Macro-F1': result.get('macro_f1'),
        'accuracy': result.get('accuracy'),
        'false_reassurance': errors.get('false_reassurance_rate'),
        'false_alarm': errors.get('false_alarm_rate'),
    })